# 04 - MAGE folds

Builds the training, calibration and test sets for every condition in §3.4, using the cleaning decided in `03`.

1. Load and clean
2. Length matching on one training set
3. All training sets
4. Calibration and test sets
5. Save

In [1]:
from logos.paths import DATA_DIR
import pandas as pd

from logos.project_data.mage import load_mage, clean_mage, families
from logos.project_data.preprocess import length_match_by_domain
from logos.evaluation.folds import training_sets, calibration_set, test_set

LABELS = {0: "human", 1: "machine"}

## 1. Load and clean
`clean_mage` applies every step from `03` in the same order, so the counts must match `03` exactly.

In [2]:
# 1. Download and clean MAGE
df = clean_mage(load_mage())

# 2. Check for the same counts as 03 (392,385 texts: 243,160 machine, 149,225 human)
print("texts:", len(df))
print(df["label"].map(LABELS).value_counts().to_string())

texts: 392385
label
machine    243160
human      149225


## 2. Length matching on one training set
The in-distribution training set is matched first, to check the matching before building every fold. Each machine text is paired with an unused human text from the same domain within ±20% of its length.

In [3]:
# 1. The training split, and the in-distribution training set matched from it
train = df[df["split"] == "train"]
matched = length_match_by_domain(train)

# 2. Texts available vs pairs kept, per domain
available = pd.crosstab(train["domain"], train["label"].map(LABELS))
retention = pd.DataFrame({
    "human available": available["human"],
    "machine available": available["machine"],
    "pairs kept": matched["domain"].value_counts() // 2,
})
retention["% of humans kept"] = (100 * retention["pairs kept"] / retention["human available"]).round(1)
print(retention.to_string())
print("total pairs:", matched["pair_id"].nunique())

         human available  machine available  pairs kept  % of humans kept
domain                                                                   
cmv                 4212              16556        4157              98.7
eli5               16663              20764       12983              77.9
hswag               2756              20262        2679              97.2
roct                3287              20712        3268              99.4
sci_gen             4433              18691        4432             100.0
squad              15820              19940        8211              51.9
tldr                2823              16091        2796              99.0
wp                  6356              20003        5932              93.3
xsum                4707              21255        4590              97.5
yelp               31509              20517       18683              59.3
total pairs: 67731


Matching keeps 67,731 pairs (135,462 texts). That is 73.2% of the 92,566 human training texts and 34.8% of the 194,791 machine ones. Retention depends on how many machine texts each human text can choose from. Domains with three or more machine texts per human keep 93.3% to 100% of their humans, even where median length differs most (hswag, roct). Retention is lowest in squad (51.9%), yelp (59.3%) and eli5 (77.9%), where human and machine counts are close. Yelp has fewer machine texts than human, so it could keep at most 65.1%. Every domain now has equal numbers of human and machine texts, which removes the link between domain and label seen in 03.

In [4]:
# 1. Median length per domain, before and after matching
before = train.groupby(["domain", train["label"].map(LABELS)])["n_words"].median().unstack()
after = matched.groupby(["domain", matched["label"].map(LABELS)])["n_words"].median().unstack()
print(before.join(after, lsuffix=" before", rsuffix=" after").to_string())

# 2. Share of each family's machine texts that found a partner
available_by_family = train.loc[train["label"] == 1, "family"].value_counts()
kept_by_family = matched.loc[matched["label"] == 1, "family"].value_counts()
print((100 * kept_by_family / available_by_family).round(1).sort_values().to_string())

label    human before  machine before  human after  machine after
domain                                                           
cmv             205.0           107.0        204.0          201.0
eli5            163.0            88.0        132.0          129.0
hswag            28.0            95.0         29.0           30.0
roct             44.0           111.0         44.0           41.0
sci_gen         160.0           145.0        160.0          157.0
squad           108.0           177.0        112.0          112.0
tldr             78.0           121.0         78.0           77.0
wp              429.0           163.0        402.0          403.0
xsum            291.0           104.0        285.0          277.0
yelp             99.0            84.0         80.0           80.0
family
GLM-130B      27.8
FLAN-T5       29.3
BigScience    29.5
LLaMA         35.5
OPT           36.4
EleutherAI    37.6
OpenAI GPT    44.2


Matching closes the length gap within every domain. Medians that differed by up to 3.4 times before matching (hswag, 28 against 95 words) now differ by at most 8 words (xsum, 285 against 277). In domains with a lot of machine texts, matching picks the machine texts that fit the human lengths. Where there are more human texts (yelp, eli5), the ones kept are those closest to machine lengths. Families keep different shares of their training texts, from 27.8% (GLM-130B) to 44.2% (OpenAI GPT). The families with the most extreme lengths (GLM-130B, FLAN-T5, BigScience) lose the most, because few human texts are as long or as short. This is the intended effect of removing length as a cue. The texts dropped are those whose length alone would mark them as machine. Every family remains represented, and the test sets keep their natural lengths.

## 3. All training sets
One in-distribution set and seven LOFO sets, one per held-out family, are built once for each of five seeds. Each seed shuffles the machine texts into a different order before matching, so different pairs are formed. In every set, a tenth of the pairs is held back as a tuning set. This set is used only to choose hyperparameters and fit the fusion, never to train a detector.

In [5]:
# 1. Build every training set, once per seed
SEEDS = [0, 1, 2, 3, 4]

all_sets = {}

for seed in SEEDS:
    all_sets[seed] = training_sets(df, seed=seed)

sets = all_sets[0]

# 2. Pairs in each set, per seed
size_rows = []

for seed, seed_sets in all_sets.items():
    for set_name, set_df in seed_sets.items():
        size_rows.append({"seed": seed, "training set": set_name, "pairs": set_df["pair_id"].nunique()})
sizes = pd.DataFrame(size_rows)

print(sizes.pivot(index="training set", columns="seed", values="pairs").to_string())

# 3. Share of pairs held back for tuning (should be 10.0 everywhere)
tune_share = []

for seed, seed_sets in all_sets.items():
    for set_name, set_df in seed_sets.items():
        tune_pairs = set_df.loc[set_df["part"] == "tune", "pair_id"].nunique()
        tune_share.append(round(100 * tune_pairs / set_df["pair_id"].nunique(), 1))

print("tune share (%):", sorted(set(tune_share)))

seed                 0      1      2      3      4
training set                                      
In-distribution  55148  55139  55159  55146  55162
LOFO BigScience  65416  65396  65437  65402  65405
LOFO EleutherAI  65617  65622  65615  65658  65656
LOFO FLAN-T5     62843  62854  62850  62885  62870
LOFO GLM-130B    66994  67017  67016  67026  66999
LOFO LLaMA       62487  62455  62483  62468  62478
LOFO OPT         55148  55139  55159  55146  55162
LOFO OpenAI GPT  63124  63145  63136  63103  63124
tune share (%): [10.0]


For seed 0, LOFO training sets range from 55,148 pairs (OPT held out) to 66,994 (GLM-130B held out). Matched on all families, the in-distribution set would have 67,731 pairs, 22.8% more than the smallest LOFO set. Part of any drop from in-distribution to LOFO could then reflect training size rather than generator shift. To rule this out, the in-distribution set is cut to the smallest LOFO size in each seed, so every LOFO set has at least as much training data as the condition it is compared with. The 67,731 pairs in section 2 are the in-distribution set before this cut. Set sizes change by less than 0.1% between seeds, and exactly 10.0% of the pairs in every set are held back for tuning.

In [9]:
# 1. No training set contains its held-out family, in any seed (should be 0)
leaks = 0

for seed, seed_sets in all_sets.items():
    for lofo_family in families:
        leaks += (seed_sets[f"LOFO {lofo_family}"]["family"] == lofo_family).sum()

print("held-out texts found in any training set:", leaks)

# 2. Do the seeds really change the pairs? Share of seed 0's in-distribution texts also used by each other seed
first = all_sets[0]["In-distribution"]

for seed in SEEDS[1:]:
    other = all_sets[seed]["In-distribution"]

    for label in [0, 1]:
        first_ids = set(first.loc[first["label"] == label, "text_id"])
        other_ids = set(other.loc[other["label"] == label, "text_id"])
        shared = 100 * len(first_ids & other_ids) / len(first_ids)
        print(f"seed {seed}, {LABELS[label]}: {shared:.1f}% shared with seed 0")

# 3. Without the in-distribution cut, are the same human texts kept in every seed?
first_lofo = all_sets[0]["LOFO OPT"]
first_humans = set(first_lofo.loc[first_lofo["label"] == 0, "text_id"])

for seed in SEEDS[1:]:
    other_lofo = all_sets[seed]["LOFO OPT"]
    other_humans = set(other_lofo.loc[other_lofo["label"] == 0, "text_id"])
    shared = 100 * len(first_humans & other_humans) / len(first_humans)
    print(f"seed {seed}, LOFO OPT humans: {shared:.1f}% shared with seed 0")

held-out texts found in any training set: 0
seed 1, human: 81.4% shared with seed 0
seed 1, machine: 57.9% shared with seed 0
seed 2, human: 81.4% shared with seed 0
seed 2, machine: 57.7% shared with seed 0
seed 3, human: 81.4% shared with seed 0
seed 3, machine: 57.6% shared with seed 0
seed 4, human: 81.4% shared with seed 0
seed 4, machine: 57.5% shared with seed 0
seed 1, LOFO OPT humans: 99.9% shared with seed 0
seed 2, LOFO OPT humans: 99.9% shared with seed 0
seed 3, LOFO OPT humans: 99.9% shared with seed 0
seed 4, LOFO OPT humans: 99.9% shared with seed 0


No training set contains texts from its held-out family, in any seed. The seeds change the training data, mostly on the machine side. Each seed reuses 57.5% to 57.9% of seed 0’s in-distribution machine texts, but 81.4% of its human texts. That 81.4% equals 55,148 / 67,731, the share kept by the in-distribution cut. Without the cut, 99.9% of the LOFO OPT human texts are the same in every seed. Human texts are the scarcer side in most domains, so nearly all are kept whatever the order. What the seeds change is which machine texts they are paired with.

## 4. Calibration and test sets
τ* is calibrated on human texts from the validation split. Every condition is tested on the test split, using its human texts plus the target’s machine texts.

In [7]:
# 1. Calibration: validation human texts only
print("calibration human texts:", len(calibration_set(df)))

# 2. Test set for each held-out family: all test humans plus that family's machine texts
test_rows = []
for lofo_family in families:
    target_df = test_set(df, target=lofo_family)
    test_rows.append({"target": lofo_family,
                      "human": (target_df["label"] == 0).sum(),
                      "machine": (target_df["label"] == 1).sum()})
print(pd.DataFrame(test_rows).to_string(index=False))

calibration human texts: 28407
    target  human  machine
OpenAI GPT  28252     2790
     LLaMA  28252     3710
  GLM-130B  28252      919
   FLAN-T5  28252     4658
       OPT  28252     8012
BigScience  28252     2692
EleutherAI  28252     1442


Calibration uses 28,407 human texts from the validation split. Every test set contains the same 28,252 human texts from the test split, plus the held-out family’s machine texts, from 919 (GLM-130B) to 8,012 (OPT). All counts match 03. Because the human test texts are identical across targets, differences in test FPR between folds come from the detectors, not the texts. Test FPR also shows whether τ*, set on validation, holds on unseen human text.

## 5. Save
The cleaned corpus is saved once. Each training set is saved for every seed as a list of text ids, with each text marked as train or tune, so detector runs load the same data without repeating the cleaning.

In [8]:
MAGE_DIR = DATA_DIR / "mage"
MAGE_DIR.mkdir(parents=True, exist_ok=True)

# 1. The cleaned corpus
df.to_parquet(MAGE_DIR / "corpus.parquet", index=False)

# 2. Which texts are in each training set: one row per seed, training set and text
membership = []

for seed, seed_sets in all_sets.items():
    for set_name, set_df in seed_sets.items():
        membership.append(pd.DataFrame({
            "seed": seed,
            "training_set": set_name,
            "text_id": set_df["text_id"],
            "pair_id": set_df["pair_id"],
            "part": set_df["part"],
        }))

pd.concat(membership).to_parquet(MAGE_DIR / "training_sets.parquet", index=False)
print("saved to", MAGE_DIR)

saved to /Users/xwm517/Desktop/logos/data/mage


## Observations

### Corpus
`clean_mage` reproduces `03` exactly, with 392,385 texts (149,225 human, 243,160 machine).

### Matching retention
 Matching the full training split keeps 67,731 pairs, 73.2% of the 92,566 human training texts. Retention depends on how many machine texts each human text can choose from. Domains with three or more keep 93.3% to 100% of their humans, while squad (51.9%), yelp (59.3%) and eli5 (77.9%) keep the fewest.

### Length
Within-domain medians that differed by up to 3.4 times now differ by at most 8 words. Every domain has equal numbers of human and machine texts, which removes the link between domain and label seen in `03`.

### Families
Families keep 27.8% (GLM-130B) to 44.2% (OpenAI GPT) of their training texts. Those with the most extreme lengths lose the most, because their length alone would mark them as machine.

### Training sets
For seed 0, LOFO sets range from 55,148 pairs (OPT held out) to 66,994 (GLM-130B held out). Sizes change by less than 0.1% between seeds. The in-distribution set is cut to the smallest LOFO size in every seed, so no comparison favours the condition with more training data. No training set contains its held-out family.

### Seeds and tuning
Each seed forms different pairs, reusing 57.5% to 57.9% of seed 0's in-distribution machine texts and 81.4% of its human texts. Every set holds back exactly 10.0% of its pairs for tuning.

### Calibration and test
τ* is calibrated on 28,407 validation human texts. Every test set shares the same 28,252 human texts, with 919 to 8,012 machine texts per held-out family.

### Saved
The cleaned corpus and the membership of all 40 training sets (8 sets × 5 seeds) are saved, so every detector run uses identical data.